# Task 1


## 1. Install packages

In [ ]:
!pip -q install pdfplumber pandas tqdm

## 2. Imports and paths

In [ ]:
import os
import re
import sqlite3
import zipfile
import shutil
from pathlib import Path

import pdfplumber
import pandas as pd
from tqdm import tqdm

PDF_FOLDER = Path('/content/pdf_folder')
DB_PATH = Path('/content/well_reports.sqlite')
ZIP_PATH = Path('/content/PDF_VERSION_1000.zip')

## 3. Upload/extract the PDF ZIP

In [ ]:
# Upload/extract the PDF folder only when needed.
# Expected input: a zip that contains PDF_VERSION_1000/ or any folder with the 1000 PDFs.
try:
    from google.colab import files
except Exception:
    files = None

if not any(PDF_FOLDER.rglob("*.pdf")):
    if not ZIP_PATH.exists():
        # Accept any uploaded zip name, e.g. PDF_version_1000.zip or PDF_VERSION_1000.zip.
        zip_candidates = sorted(Path("/content").glob("*.zip"))
        if zip_candidates:
            ZIP_PATH = zip_candidates[0]

    if not ZIP_PATH.exists():
        if files is None:
            raise FileNotFoundError("No PDFs found and no zip found. Put the PDFs in PDF_FOLDER or upload a zip.")
        uploaded = files.upload()
        ZIP_PATH = Path(next(iter(uploaded)))

    shutil.rmtree(PDF_FOLDER, ignore_errors=True)
    PDF_FOLDER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(PDF_FOLDER)

pdf_files = sorted(PDF_FOLDER.rglob("*.pdf"))
print("ZIP used:", ZIP_PATH if ZIP_PATH.exists() else "already extracted")
print("PDF folder:", PDF_FOLDER)
print("PDFs found:", len(pdf_files))
print("Database will be saved to:", DB_PATH)

## 4. Text cleaning helpers

In [ ]:
def undouble_token(token: str) -> str:
    """Fix duplicated bold text such as WWeellllbboorree -> Wellbore."""
    if not token or not re.search(r"[A-Za-z]", token):
        return token
    if len(token) >= 4 and len(token) % 2 == 0:
        pairs = [token[i:i+2] for i in range(0, len(token), 2)]
        if all(len(p) == 2 and p[0] == p[1] for p in pairs):
            return ''.join(p[0] for p in pairs)
    return token


def undouble_text(text: str) -> str:
    return re.sub(r"\S+", lambda m: undouble_token(m.group(0)), text or "")


def clean_cell(value) -> str:
    if value is None:
        return ""
    value = undouble_text(str(value))
    value = value.replace("\n", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def clean_activity(value: str) -> str:
    value = clean_cell(value).lower()
    # Repair common PDF line-break artifacts in activity labels.
    replacements = {
        r"circu\s+lating\s+conditio\s+ning": "circulating conditioning",
        r"circu\s+lating": "circulating",
        r"conditio\s+ning": "conditioning",
        r"tripp\s+ing": "tripping",
        r"drill\s+ing": "drilling",
        r"wait\s+ing": "waiting",
        r"cement\s+ing": "cementing",
        r"comple\s+tion": "completion",
        r"ca\s+sing": "casing",
    }
    for pattern, repl in replacements.items():
        value = re.sub(pattern, repl, value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def to_float(value):
    value = clean_cell(value)
    if value == "":
        return None
    value = value.replace(",", ".")
    try:
        x = float(value)
    except Exception:
        return None
    # These reports use -999.99 as a missing-value placeholder.
    if abs(x + 999.99) < 1e-9:
        return None
    return x


def to_bool(value):
    value = clean_cell(value).upper()
    if value in {"Y", "YES", "TRUE", "1"}:
        return 1
    if value in {"N", "NO", "FALSE", "0"}:
        return 0
    return None


def normalize_key(key: str) -> str:
    key = clean_cell(key)
    key = key.rstrip(":").strip()
    key = re.sub(r"\s+", " ", key)
    return key


def key_id(key: str) -> str:
    """Strong label normalization: removes units/punctuation, keeps words/numbers."""
    key = normalize_key(key).lower()
    key = re.sub(r"\([^)]*\)", " ", key)           # remove units like (m), (g/cm3)
    key = key.replace("mmd", "m md").replace("mtvd", "m tvd")
    key = re.sub(r"[^a-z0-9]+", " ", key)
    return re.sub(r"\s+", " ", key).strip()


def get_field(fields: dict, possible_names):
    """Return the first matching field value using exact, prefix, then contains matching."""
    indexed = [(key_id(k), v) for k, v in fields.items() if clean_cell(v) != ""]
    wanted = [key_id(name) for name in possible_names]

    for name in wanted:
        for k, v in indexed:
            if k == name:
                return v
    for name in wanted:
        for k, v in indexed:
            if k.startswith(name) or name.startswith(k):
                return v
    for name in wanted:
        for k, v in indexed:
            if name in k or k in name:
                return v
    return None


def get_by_position(row, index):
    return clean_cell(row[index]) if index < len(row) else ""

## 5. Database schema

In [ ]:
def create_database(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("PRAGMA foreign_keys = ON;")
    cur.executescript("""
    DROP TABLE IF EXISTS extraction_errors;
    DROP TABLE IF EXISTS other_sections;
    DROP TABLE IF EXISTS drilling_fluids;
    DROP TABLE IF EXISTS summary_sections;
    DROP TABLE IF EXISTS equipment_failures;
    DROP TABLE IF EXISTS operations;
    DROP TABLE IF EXISTS report_depths;
    DROP TABLE IF EXISTS reports;

    CREATE TABLE reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT NOT NULL,
        file_path TEXT,
        raw_text TEXT,
        wellbore_id TEXT,
        report_number TEXT,
        period_start TEXT,
        period_end TEXT,
        status TEXT,
        operator TEXT,
        rig_name TEXT,
        drilling_contractor TEXT,
        spud_date TEXT,
        water_depth_msl REAL,
        elevation_rkb_msl REAL,
        tight_well_flag INTEGER,
        hpht_flag INTEGER,
        formation_strength REAL,
        hole_diameter REAL,
        pressure_test_type TEXT
    );

    CREATE TABLE report_depths (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        depth_type TEXT NOT NULL,
        depth_mmd REAL,
        depth_mtvd REAL,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE operations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        start_time TEXT,
        end_time TEXT,
        end_depth_mmd REAL,
        main_activity TEXT,
        sub_activity TEXT,
        state TEXT,
        remark TEXT,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE equipment_failures (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        start_time TEXT,
        depth_mmd REAL,
        depth_mtvd REAL,
        equipment_system TEXT,
        equipment_class TEXT,
        downtime_minutes REAL,
        remark TEXT,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE summary_sections (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        section_type TEXT NOT NULL,
        section_text TEXT,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE drilling_fluids (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        fluid_type TEXT,
        density REAL,
        plastic_viscosity REAL,
        yield_point REAL,
        sample_time TEXT,
        sample_depth_mmd REAL,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE other_sections (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        report_id INTEGER NOT NULL,
        section_name TEXT,
        section_text TEXT,
        FOREIGN KEY(report_id) REFERENCES reports(id)
    );

    CREATE TABLE extraction_errors (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT,
        error_message TEXT
    );
    """)
    conn.commit()
    conn.close()

## 6. Table detection helpers

In [ ]:
def clean_table(table):
    return [[clean_cell(cell) for cell in row] for row in (table or []) if row]


def table_to_text(table):
    rows = []
    for row in clean_table(table):
        if any(row):
            rows.append(" | ".join(row))
    return "\n".join(rows)


def table_text(table):
    return " ".join(" ".join(row) for row in clean_table(table)).lower()


def get_header(table):
    rows = clean_table(table)
    return " | ".join(rows[0]).lower() if rows else ""


def is_operations_table(table):
    header = get_header(table)
    return (
        "start" in header
        and "end" in header
        and "depth" in header
        and "remark" in header
        and ("activity" in header or "main" in header)
    )


def is_equipment_failure_table(table):
    txt = table_text(table)
    return (
        "equipment" in txt
        and "downtime" in txt
        and "remark" in txt
        and "depth" in txt
    )


def is_drilling_fluid_table(table):
    txt = table_text(table)
    first_col = " ".join(row[0].lower() for row in clean_table(table) if row)
    return (
        "fluid type" in txt
        and ("fluid density" in txt or "density" in txt)
        and ("sample time" in first_col or "sample depth" in first_col or "plastic visc" in txt or "yield point" in txt)
    )


def is_key_value_table(table):
    rows = clean_table(table)
    if not rows:
        return False
    usable = 0
    good = 0
    for row in rows:
        if len(row) < 2:
            continue
        key = row[0]
        value = row[1]
        if key:
            usable += 1
            # Header metadata rows have short labels and value cells.
            if len(key) < 90 and (value != "" or key.endswith(":") or re.search(r":$", key)):
                good += 1
    return usable > 0 and good / usable >= 0.55

## 7. Row extraction helpers

In [ ]:
def split_main_sub(value):
    value = clean_activity(value)
    parts = re.split(r"\s+--\s+|\s+-\s+", value, maxsplit=1)
    main = parts[0].strip() if parts else ""
    sub = parts[1].strip() if len(parts) > 1 else ""
    return main, sub


def extract_operations_from_table(table):
    operations = []
    rows = clean_table(table)
    for row in rows[1:]:
        if not any(row):
            continue
        # Most PDFs have 6 columns: start, end, depth, combined main--sub, state, remark.
        # Some may already have separate main/sub columns. Support both.
        if len(row) >= 7:
            main_activity = clean_activity(get_by_position(row, 3))
            sub_activity = clean_activity(get_by_position(row, 4))
            state = get_by_position(row, 5).lower()
            remark = get_by_position(row, 6)
        else:
            main_activity, sub_activity = split_main_sub(get_by_position(row, 3))
            state = get_by_position(row, 4).lower()
            remark = get_by_position(row, 5)

        operations.append({
            "start_time": get_by_position(row, 0),
            "end_time": get_by_position(row, 1),
            "end_depth_mmd": to_float(get_by_position(row, 2)),
            "main_activity": main_activity,
            "sub_activity": sub_activity,
            "state": state,
            "remark": remark,
        })
    return operations


def split_equipment(value):
    value = clean_activity(value)
    parts = re.split(r"\s+--\s+|\s+-\s+", value, maxsplit=1)
    system = parts[0].strip() if parts else ""
    equipment_class = parts[1].strip() if len(parts) > 1 else ""
    return system, equipment_class


def extract_equipment_failures_from_table(table):
    failures = []
    rows = clean_table(table)
    for row in rows[1:]:
        if not any(row):
            continue
        # Actual layout: start, depth mMD, depth mTVD, combined equipment system/class, downtime, repaired, remark.
        system, equipment_class = split_equipment(get_by_position(row, 3))
        downtime = to_float(get_by_position(row, 4))
        repaired = get_by_position(row, 5)
        remark = get_by_position(row, 6)
        if repaired:
            remark = f"Equipment repaired: {repaired}. {remark}".strip()
        failures.append({
            "start_time": get_by_position(row, 0),
            "depth_mmd": to_float(get_by_position(row, 1)),
            "depth_mtvd": to_float(get_by_position(row, 2)),
            "equipment_system": system,
            "equipment_class": equipment_class,
            "downtime_minutes": downtime,
            "remark": remark,
        })
    return failures


def extract_drilling_fluids_from_table(table):
    rows = clean_table(table)
    if not rows:
        return []
    width = max(len(row) for row in rows)
    fluids = []
    # Fluid tables are transposed: first column = property, remaining columns = samples.
    for col in range(1, width):
        sample = {
            "fluid_type": "",
            "density": None,
            "plastic_viscosity": None,
            "yield_point": None,
            "sample_time": "",
            "sample_depth_mmd": None,
        }
        has_value = False
        for row in rows:
            label = key_id(get_by_position(row, 0))
            value = get_by_position(row, col)
            if value:
                has_value = True
            if label == "sample time":
                sample["sample_time"] = value
            elif label == "sample depth m md":
                sample["sample_depth_mmd"] = to_float(value)
            elif label == "fluid type":
                sample["fluid_type"] = value
            elif label == "fluid density":
                sample["density"] = to_float(value)
            elif label.startswith("plastic visc"):
                sample["plastic_viscosity"] = to_float(value)
            elif label == "yield point":
                sample["yield_point"] = to_float(value)
        if has_value and (sample["fluid_type"] or sample["density"] is not None or sample["sample_time"]):
            fluids.append(sample)
    return fluids

## 8. PDF-level extraction

In [ ]:
def extract_summary_sections(text):
    summaries = []
    activity_match = re.search(
        r"Summary of activities\s*\(24 Hours\)\s*(.*?)\s*Summary of planned activities\s*\(24 Hours\)",
        text,
        re.S | re.I,
    )
    if activity_match:
        summaries.append({"section_type": "ACTIVITIES_24H", "section_text": activity_match.group(1).strip()})

    planned_match = re.search(
        r"Summary of planned activities\s*\(24 Hours\)\s*(.*?)(Survey Station|DDeepptthh|Depth mMD|$)",
        text,
        re.S | re.I,
    )
    if planned_match:
        summaries.append({"section_type": "PLANNED_ACTIVITIES_24H", "section_text": planned_match.group(1).strip()})
    return summaries


def extract_wellbore_and_period(text):
    m = re.search(
        r"Wellbore:\s*(.*?)\s+Period:\s*"
        r"(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2})\s*-\s*"
        r"(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2})",
        text,
        re.I,
    )
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    return None, None, None


def add_fields_from_key_value_table(fields: dict, table):
    for row in clean_table(table):
        if len(row) < 2:
            continue
        key = normalize_key(row[0])
        value = clean_cell(row[1])
        if key and key.lower() not in {"summary report", ""}:
            fields[key] = value


def add_fields_from_text(fields: dict, text: str):
    # Fallback extraction for labels that are sometimes missed by table parsing.
    labels = [
        "Status", "Report creation time", "Report number", "Days Ahead/Behind (+/-)",
        "Operator", "Rig Name", "Drilling contractor", "Spud Date", "Wellbore type",
        "Elevation RKB-MSL (m)", "Water depth MSL (m)", "Tight well", "HPHT",
        "Dist Drilled (m)", "Penetration rate (m/h)", "Hole Dia (in)", "Pressure Test Type",
        "Formation strength (g/cm3)", "Plug Back Depth mMD", "Depth mMd", "Depth mTVD",
        "Depth at Kick Off mMD", "Depth at Kick Off mTVD",
        "Depth At Last Casing mMD", "Depth At Last Casing mTVD",
        "Depth at formation strength mMD", "Depth At Formation Strength mTVD",
    ]
    label_re = "|".join(re.escape(x) for x in sorted(labels, key=len, reverse=True))
    pattern = re.compile(rf"({label_re}):\s*(.*?)(?=\s+(?:{label_re}):|\n|$)", re.I)
    for m in pattern.finditer(text):
        key = normalize_key(m.group(1))
        value = clean_cell(m.group(2))
        if key and value and key not in fields:
            fields[key] = value


def extract_pdf(pdf_path):
    all_text = []
    fields = {}
    operations = []
    equipment_failures = []
    drilling_fluids = []
    other_sections = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = undouble_text(page.extract_text() or "")
            all_text.append(page_text)
            tables = page.extract_tables() or []
            for table_index, table in enumerate(tables, start=1):
                if not table:
                    continue
                if is_operations_table(table):
                    operations.extend(extract_operations_from_table(table))
                elif is_equipment_failure_table(table):
                    equipment_failures.extend(extract_equipment_failures_from_table(table))
                elif is_drilling_fluid_table(table):
                    drilling_fluids.extend(extract_drilling_fluids_from_table(table))
                elif is_key_value_table(table):
                    add_fields_from_key_value_table(fields, table)
                else:
                    other_sections.append({
                        "section_name": f"Page {page_number} Table {table_index}",
                        "section_text": table_to_text(table),
                    })

    text = "\n".join(all_text)
    add_fields_from_text(fields, text)
    wellbore_id, period_start, period_end = extract_wellbore_and_period(text)

    return {
        "filename": Path(pdf_path).name,
        "file_path": str(pdf_path),
        "raw_text": text,
        "wellbore_id": wellbore_id,
        "report_number": get_field(fields, ["Report number"]),
        "period_start": period_start,
        "period_end": period_end,
        "status": get_field(fields, ["Status"]),
        "operator": get_field(fields, ["Operator"]),
        "rig_name": get_field(fields, ["Rig Name"]),
        "drilling_contractor": get_field(fields, ["Drilling contractor"]),
        "spud_date": get_field(fields, ["Spud Date"]),
        "water_depth_msl": to_float(get_field(fields, ["Water depth MSL", "Water depth"])),
        "elevation_rkb_msl": to_float(get_field(fields, ["Elevation RKB MSL", "Elevation RKB-MSL"])),
        "tight_well_flag": to_bool(get_field(fields, ["Tight well"])),
        "hpht_flag": to_bool(get_field(fields, ["HPHT"])),
        "formation_strength": to_float(get_field(fields, ["Formation strength"])),
        "hole_diameter": to_float(get_field(fields, ["Hole Dia", "Hole diameter"])),
        "pressure_test_type": get_field(fields, ["Pressure Test Type", "Pressure test type"]),
        "depths": [
            {"depth_type": "CURRENT", "depth_mmd": to_float(get_field(fields, ["Depth mMd", "Depth mMD", "Current depth mMD"])), "depth_mtvd": to_float(get_field(fields, ["Depth mTVD", "Current depth mTVD"]))},
            {"depth_type": "KICK_OFF", "depth_mmd": to_float(get_field(fields, ["Depth at Kick Off mMD"])), "depth_mtvd": to_float(get_field(fields, ["Depth at Kick Off mTVD"]))},
            {"depth_type": "LAST_CASING", "depth_mmd": to_float(get_field(fields, ["Depth At Last Casing mMD", "Depth at Last Casing mMD"])), "depth_mtvd": to_float(get_field(fields, ["Depth At Last Casing mTVD", "Depth at Last Casing mTVD"]))},
            {"depth_type": "FORMATION_STRENGTH", "depth_mmd": to_float(get_field(fields, ["Depth at formation strength mMD", "Depth At Formation Strength mMD"])), "depth_mtvd": to_float(get_field(fields, ["Depth At Formation Strength mTVD", "Depth at formation strength mTVD"]))},
            {"depth_type": "PLUG_BACK", "depth_mmd": to_float(get_field(fields, ["Plug Back Depth mMD"])), "depth_mtvd": None},
        ],
        "operations": operations,
        "equipment_failures": equipment_failures,
        "summary_sections": extract_summary_sections(text),
        "drilling_fluids": drilling_fluids,
        "other_sections": [s for s in other_sections if s["section_text"].strip()],
    }

## 9. Insert data and process all PDFs

In [ ]:
def insert_extracted_report(conn, data):
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO reports (
            filename, file_path, raw_text, wellbore_id, report_number,
            period_start, period_end, status, operator, rig_name,
            drilling_contractor, spud_date, water_depth_msl, elevation_rkb_msl,
            tight_well_flag, hpht_flag, formation_strength, hole_diameter,
            pressure_test_type
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        data["filename"], data["file_path"], data["raw_text"], data["wellbore_id"],
        data["report_number"], data["period_start"], data["period_end"], data["status"],
        data["operator"], data["rig_name"], data["drilling_contractor"], data["spud_date"],
        data["water_depth_msl"], data["elevation_rkb_msl"], data["tight_well_flag"],
        data["hpht_flag"], data["formation_strength"], data["hole_diameter"],
        data["pressure_test_type"],
    ))
    report_id = cur.lastrowid

    for depth in data["depths"]:
        if depth["depth_mmd"] is not None or depth["depth_mtvd"] is not None:
            cur.execute("""
                INSERT INTO report_depths (report_id, depth_type, depth_mmd, depth_mtvd)
                VALUES (?, ?, ?, ?)
            """, (report_id, depth["depth_type"], depth["depth_mmd"], depth["depth_mtvd"]))

    for op in data["operations"]:
        if op["remark"] or op["main_activity"] or op["sub_activity"]:
            cur.execute("""
                INSERT INTO operations (
                    report_id, start_time, end_time, end_depth_mmd,
                    main_activity, sub_activity, state, remark
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                report_id, op["start_time"], op["end_time"], op["end_depth_mmd"],
                op["main_activity"], op["sub_activity"], op["state"], op["remark"],
            ))

    for failure in data["equipment_failures"]:
        cur.execute("""
            INSERT INTO equipment_failures (
                report_id, start_time, depth_mmd, depth_mtvd,
                equipment_system, equipment_class, downtime_minutes, remark
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            report_id, failure["start_time"], failure["depth_mmd"], failure["depth_mtvd"],
            failure["equipment_system"], failure["equipment_class"],
            failure["downtime_minutes"], failure["remark"],
        ))

    for summary in data["summary_sections"]:
        if summary["section_text"]:
            cur.execute("""
                INSERT INTO summary_sections (report_id, section_type, section_text)
                VALUES (?, ?, ?)
            """, (report_id, summary["section_type"], summary["section_text"]))

    for fluid in data["drilling_fluids"]:
        cur.execute("""
            INSERT INTO drilling_fluids (
                report_id, fluid_type, density, plastic_viscosity,
                yield_point, sample_time, sample_depth_mmd
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            report_id, fluid["fluid_type"], fluid["density"], fluid["plastic_viscosity"],
            fluid["yield_point"], fluid["sample_time"], fluid["sample_depth_mmd"],
        ))

    for section in data["other_sections"]:
        cur.execute("""
            INSERT INTO other_sections (report_id, section_name, section_text)
            VALUES (?, ?, ?)
        """, (report_id, section["section_name"], section["section_text"]))


def process_all_pdfs(pdf_folder, db_path):
    pdf_files = sorted(Path(pdf_folder).rglob("*.pdf"))
    create_database(db_path)
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()
    ok = 0
    errors = 0
    for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
        try:
            data = extract_pdf(pdf_path)
            insert_extracted_report(conn, data)
            ok += 1
        except Exception as e:
            errors += 1
            cur.execute("INSERT INTO extraction_errors (filename, error_message) VALUES (?, ?)", (Path(pdf_path).name, str(e)))
    conn.commit()
    conn.close()
    return {"pdfs_found": len(pdf_files), "pdfs_processed": ok, "errors": errors, "db_path": str(db_path)}

## 10. Run extraction

In [ ]:
result = process_all_pdfs(PDF_FOLDER, DB_PATH)
result

## 11. Validate database counts

In [ ]:
conn = sqlite3.connect(DB_PATH)

counts = pd.read_sql("""
SELECT 'reports' AS table_name, COUNT(*) AS rows FROM reports UNION ALL
SELECT 'report_depths', COUNT(*) FROM report_depths UNION ALL
SELECT 'operations', COUNT(*) FROM operations UNION ALL
SELECT 'equipment_failures', COUNT(*) FROM equipment_failures UNION ALL
SELECT 'summary_sections', COUNT(*) FROM summary_sections UNION ALL
SELECT 'drilling_fluids', COUNT(*) FROM drilling_fluids UNION ALL
SELECT 'other_sections', COUNT(*) FROM other_sections UNION ALL
SELECT 'extraction_errors', COUNT(*) FROM extraction_errors
""", conn)
counts

## 12. Quality checks for important fixes

In [ ]:
quality = pd.read_sql("""
SELECT
    COUNT(*) AS report_count,
    SUM(CASE WHEN wellbore_id IS NULL OR TRIM(wellbore_id) = '' THEN 1 ELSE 0 END) AS missing_wellbore_id,
    SUM(CASE WHEN report_number IS NULL OR TRIM(report_number) = '' THEN 1 ELSE 0 END) AS missing_report_number,
    SUM(CASE WHEN elevation_rkb_msl IS NULL THEN 1 ELSE 0 END) AS missing_elevation_rkb_msl,
    SUM(CASE WHEN formation_strength IS NULL THEN 1 ELSE 0 END) AS missing_formation_strength,
    SUM(CASE WHEN hole_diameter IS NULL THEN 1 ELSE 0 END) AS missing_hole_diameter
FROM reports
""", conn)

operation_quality = pd.read_sql("""
SELECT
    COUNT(*) AS operation_rows,
    SUM(CASE WHEN remark IS NULL OR TRIM(remark) = '' THEN 1 ELSE 0 END) AS blank_operation_remarks,
    SUM(CASE WHEN state IS NULL OR TRIM(state) = '' THEN 1 ELSE 0 END) AS blank_states
FROM operations
""", conn)

display(quality)
display(operation_quality)

## 13. Inspect sample operation rows

In [ ]:
pd.read_sql("""
SELECT
    r.filename,
    o.start_time,
    o.end_time,
    o.end_depth_mmd,
    o.main_activity,
    o.sub_activity,
    o.state,
    SUBSTR(o.remark, 1, 180) AS remark_preview
FROM operations o
JOIN reports r ON r.id = o.report_id
WHERE o.remark IS NOT NULL AND TRIM(o.remark) <> ''
LIMIT 10
""", conn)

## 14. Download SQLite database

In [ ]:
from google.colab import files
files.download(str(DB_PATH))